# JuriBench — πειράματα στο Colab

Fine-tuning (QLoRA) ενός μικρού μοντέλου στο JuriBench και σύγκριση με το ίδιο μοντέλο χωρίς εκπαίδευση.
Τρέχει στο δωρεάν Colab (GPU T4).

Πριν ξεκινήσεις:
1. Runtime → Change runtime type → **T4 GPU**.
2. Ανέβασε το `data.zip` (ή το `juribench_labeled.dedup.csv`) στον φάκελο `DRIVE` του κελιού 2.
3. Για Llama: αποδοχή άδειας στη σελίδα του μοντέλου στο Hugging Face και ένα access token.

Τα αποτελέσματα (adapters, προβλέψεις, μετρικές, `experiments.csv`) γράφονται στο Drive.

## 1. Παράμετροι πειράματος
Αλλάζεις μόνο αυτό το κελί και ξανατρέχεις από εδώ και κάτω.

In [ ]:
# Μοντέλο (Llama χρειάζεται HF token· το Qwen όχι)
MODEL = "meta-llama/Llama-3.1-8B-Instruct"     # χωράει στο T4 (4-bit), αλλά είναι αργό
# MODEL = "meta-llama/Llama-3.2-1B-Instruct"   # ξεχωριστή άδεια στο HF
# MODEL = "meta-llama/Llama-3.2-3B-Instruct"
# MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

TASK = "predict"          # "predict" (μόνο έκβαση) ή "generation" (ολόκληρη απόφαση, πολύ πιο αργό)

# Χωρισμός ολόκληρου του dataset
TEST_FRAC, VAL_FRAC, SEED = 0.10, 0.10, 42

# Πόσα δείγματα χρησιμοποιεί το πείραμα (τυχαία, με τις ίδιες αναλογίες δικαστηρίου × έκβασης)
N_TRAIN, N_VAL, N_TEST = 500, 50, 200   # 8B στο T4: ξεκίνα μικρά

# Εκπαίδευση
EPOCHS = 1
MAX_STEPS = -1            # >0 για να κοπεί νωρίτερα (π.χ. 50 για γρήγορη δοκιμή)
MAX_SEQ_LEN = 1024        # predict: 1024 αρκεί · generation: 2048
MAX_TARGET_CHARS = 6000   # μόνο generation: κόβει την απόφαση-στόχο
LR, LORA_R, LORA_ALPHA = 2e-4, 16, 32

RUN_BASELINE = True       # αξιολόγηση και του μοντέλου χωρίς fine-tuning

# Μόνο για generation (δοκιμές· το dataset δεν αλλάζει)
STRIP_HEADER = True       # ο στόχος ξεκινά από το κυρίως κείμενο, χωρίς κεφαλίδα (caption, σύνθεση, δικηγόροι)
REP_PENALTY = 1.15        # ποινή επανάληψης στην παραγωγή (1.0 = καμία)
NO_REPEAT_NGRAM = 8       # απαγορεύει επανάληψη ίδιας σειράς 8 tokens (0 = καμία)

TAG = MODEL.split("/")[-1].lower()
RUN = f"{TAG}_{TASK}_n{N_TRAIN}_e{EPOCHS}_s{SEED}"
if TASK == "generation":
    RUN += f"_sh{int(STRIP_HEADER)}_rp{REP_PENALTY}_ng{NO_REPEAT_NGRAM}"
print(RUN)

## 2. Setup
Σύνδεση με το Drive, λήψη του κώδικα από το GitHub, εγκατάσταση βιβλιοθηκών.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/University/Διπλωματική/JuriBench"

import os, subprocess
os.makedirs(DRIVE, exist_ok=True)
%cd /content
!rm -rf JuriBench-Thesis-NTUA
!git clone -q https://github.com/angeloskarasavvidis/JuriBench-Thesis-NTUA.git
%cd /content/JuriBench-Thesis-NTUA
!git log --oneline -1

# τα αποτελέσματα πάνε στο Drive
for d in ["adapters", "preds", "results"]:
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
    os.system(f'ln -sfn "{DRIVE}/{d}" {d}')
os.makedirs("data", exist_ok=True)

!pip install -q -r requirements-train.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Hugging Face login (μόνο για Llama)

In [ ]:
if MODEL.startswith("meta-llama/"):
    from huggingface_hub import login
    login()

## 4. Δεδομένα
Βρίσκει το dataset στο Drive, φτιάχνει τον χωρισμό train/val/test και τα αρχεία εκπαίδευσης.

In [ ]:
import glob, zipfile
CSV = "/content/juribench_labeled.dedup.csv"
if not os.path.exists(CSV):
    if os.path.exists(f"{DRIVE}/juribench_labeled.dedup.csv"):
        !cp "{DRIVE}/juribench_labeled.dedup.csv" /content/
    elif os.path.exists(f"{DRIVE}/data.zip"):
        zipfile.ZipFile(f"{DRIVE}/data.zip").extract("juribench_labeled.dedup.csv", "/content")
    else:
        raise SystemExit(f"Δεν βρέθηκε data.zip ή juribench_labeled.dedup.csv στο {DRIVE}")

!python code/make_splits.py --input {CSV} --outdir data/splits --test {TEST_FRAC} --val {VAL_FRAC} --seed {SEED}
MTC = MAX_TARGET_CHARS if TASK == "generation" else 0
SH = "--strip-header" if (TASK == "generation" and STRIP_HEADER) else ""
!python code/build_finetune.py --task {TASK} --splits data/splits --outdir data/finetune --max-target-chars {MTC} {SH}

## 5. Υποσύνολο για το πείραμα
Τυχαία επιλογή (σταθερό seed) με τις ίδιες αναλογίες δικαστηρίου × έκβασης όπως το κάθε σύνολο.
Στο predict αφαιρούνται οι υποθέσεις με έκβαση `unclear`.

In [ ]:
import json, random
from collections import defaultdict, Counter

def stratified_sample(rows, n, seed):
    if n <= 0 or n >= len(rows):
        rows = rows[:]; random.Random(seed).shuffle(rows); return rows
    groups = defaultdict(list)
    for r in rows:
        groups[(r["meta"]["jurisdiction"], r["meta"]["disposition"])].append(r)
    rng = random.Random(seed)
    quota = {k: n * len(g) / len(rows) for k, g in groups.items()}
    take = {k: int(q) for k, q in quota.items()}
    for k in sorted(quota, key=lambda k: quota[k] - take[k], reverse=True)[: n - sum(take.values())]:
        take[k] += 1
    out = []
    for k in sorted(groups):
        g = groups[k][:]; rng.shuffle(g); out += g[: take[k]]
    rng.shuffle(out)
    return out

EXP = "data/finetune_exp"
os.makedirs(f"{EXP}/{TASK}", exist_ok=True)
for split, n in [("train", N_TRAIN), ("val", N_VAL), ("test", N_TEST)]:
    rows = [json.loads(l) for l in open(f"data/finetune/{TASK}/{split}.jsonl")]
    if TASK == "predict":
        rows = [r for r in rows if r["meta"]["disposition"] != "unclear"]
    sub = stratified_sample(rows, n, SEED)
    with open(f"{EXP}/{TASK}/{split}.jsonl", "w") as f:
        for r in sub:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"{split:5} {len(sub):5} / {len(rows):5}  ", dict(Counter(r["meta"]["jurisdiction"] for r in sub)))

## 6. Baseline: το μοντέλο χωρίς fine-tuning

In [ ]:
if RUN_BASELINE:
    GEN_OPTS = f"--repetition-penalty {REP_PENALTY} --no-repeat-ngram {NO_REPEAT_NGRAM}" if TASK == "generation" else ""
    !python code/generate.py --model {MODEL} --task {TASK} --data-dir {EXP} --out preds/{RUN}_base.jsonl {GEN_OPTS}
    !python code/evaluate.py --preds preds/{RUN}_base.jsonl --task {TASK}

## 7. Fine-tuning (QLoRA)

In [ ]:
!python code/train_qlora.py --model {MODEL} --task {TASK} --data-dir {EXP} --out adapters/{RUN} \
    --epochs {EPOCHS} --max-steps {MAX_STEPS} --max-seq-len {MAX_SEQ_LEN} \
    --lr {LR} --lora-r {LORA_R} --lora-alpha {LORA_ALPHA}

## 8. Αξιολόγηση του fine-tuned μοντέλου

In [ ]:
GEN_OPTS = f"--repetition-penalty {REP_PENALTY} --no-repeat-ngram {NO_REPEAT_NGRAM}" if TASK == "generation" else ""
!python code/generate.py --model {MODEL} --adapter adapters/{RUN} --task {TASK} --data-dir {EXP} --out preds/{RUN}.jsonl {GEN_OPTS}
!python code/evaluate.py --preds preds/{RUN}.jsonl --task {TASK}

## 9. Καταγραφή και σύγκριση
Κάθε πείραμα προστίθεται ως γραμμή στο `experiments.csv` στο Drive.

In [ ]:
import pandas as pd, datetime
LOG = f"{DRIVE}/experiments.csv"
rows = []
for name, kind in [(f"{RUN}_base", "base"), (RUN, "finetuned")]:
    p = f"results/{name}.json"
    if not os.path.exists(p):
        continue
    o = json.load(open(p))["outcome"]
    rows.append(dict(date=datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), run=RUN, kind=kind,
                     model=MODEL, task=TASK, n_train=N_TRAIN, n_test=o["n"], epochs=EPOCHS,
                     max_steps=MAX_STEPS, lr=LR, lora_r=LORA_R, seed=SEED,
                     accuracy=o["accuracy"], macro_f1=o["macro_f1"],
                     **{f"acc_{k}": v for k, v in o["accuracy_by_court"].items()}))
new = pd.DataFrame(rows)
log = pd.concat([pd.read_csv(LOG), new]) if os.path.exists(LOG) else new
log.to_csv(LOG, index=False)
log.tail(10)